In [9]:
%%writefile app_logic.py
import requests
import sqlite3
import pandas as pd

# Your Harvard Art Museums API Key (provided by user)
API_KEY = "8c647885-c2b1-4554-9be1-7d8937e6ff53"
BASE_URL = "https://api.harvardartmuseums.org/object"
DB_NAME = "harvard_artifacts.db"
RECORD_LIMIT = 2500 # Set a manageable limit for quick Colab demonstrations

# --- 1. Database Initialization (Create Tables) ---
def create_tables():
    """Sets up the three required SQL tables."""
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()

    # Drop tables first for clean runs
    cursor.execute("DROP TABLE IF EXISTS artifact_metadata")
    cursor.execute("""
        CREATE TABLE artifact_metadata (
            id INTEGER PRIMARY KEY,
            title TEXT, culture TEXT, period TEXT, century TEXT, medium TEXT,
            dimensions TEXT, description TEXT, department TEXT, classification TEXT,
            accessionyear INTEGER, accessionmethod TEXT
        )
    """)
    cursor.execute("DROP TABLE IF EXISTS artifact_media")
    cursor.execute("""
        CREATE TABLE artifact_media (
            objectid INTEGER, imagecount INTEGER, mediacount INTEGER,
            colorcount INTEGER, rank INTEGER, datebegin INTEGER, dateend INTEGER,
            FOREIGN KEY (objectid) REFERENCES artifact_metadata (id)
        )
    """)
    cursor.execute("DROP TABLE IF EXISTS artifact_colors")
    cursor.execute("""
        CREATE TABLE artifact_colors (
            objectid INTEGER, color TEXT, spectrum TEXT, hue TEXT, percent REAL, css3 TEXT,
            FOREIGN KEY (objectid) REFERENCES artifact_metadata (id)
        )
    """)
    conn.commit()
    conn.close()

# --- 2. Data Extraction and Transformation (ETL) ---
def fetch_artifacts(classification_name):
    """Fetches artifacts for a given classification, handling pagination."""
    all_records = []
    page = 1
    total_records_fetched = 0
    max_pages = RECORD_LIMIT // 100 + 1

    params = {
        'apikey': API_KEY,
        'classification': classification_name,
        'size': 100,
        'fields': 'id,title,culture,period,century,medium,dimensions,description,department,classification,accessionyear,accessionmethod,imagecount,mediacount,colorcount,rank,datebegin,dateend,colors'
    }

    while total_records_fetched < RECORD_LIMIT and page <= max_pages:
        params['page'] = page
        try:
            response = requests.get(BASE_URL, params=params)
            response.raise_for_status()
            data = response.json()
        except requests.exceptions.RequestException as e:
            return [], [], [], str(e)

        records_on_page = data.get('records', [])
        if not records_on_page: break

        num_to_add = min(len(records_on_page), RECORD_LIMIT - total_records_fetched)
        all_records.extend(records_on_page[:num_to_add])
        total_records_fetched += num_to_add
        page += 1

        if total_records_fetched >= RECORD_LIMIT: break
        if page > data.get('info', {}).get('pages', page): break


    # Transformation (Splitting data into 3 tables)
    metadata_list = []
    media_list = []
    colors_list = []

    for record in all_records:
        artifact_id = record.get('id')
        if not artifact_id: continue

        # 1. artifact_metadata
        metadata_list.append({
            'id': artifact_id, 'title': record.get('title'), 'culture': record.get('culture'), 'period': record.get('period'),
            'century': record.get('century'), 'medium': record.get('medium'), 'dimensions': record.get('dimensions'),
            'description': record.get('description'), 'department': record.get('department'), 'classification': record.get('classification'),
            'accessionyear': record.get('accessionyear'), 'accessionmethod': record.get('accessionmethod')
        })

        # 2. artifact_media
        media_list.append({
            'objectid': artifact_id, 'imagecount': record.get('imagecount'), 'mediacount': record.get('mediacount'),
            'colorcount': record.get('colorcount'), 'rank': record.get('rank'), 'datebegin': record.get('datebegin'),
            'dateend': record.get('dateend')
        })

        # 3. artifact_colors
        for color_item in record.get('colors', []):
            colors_list.append({
                'objectid': artifact_id, 'color': color_item.get('color'), 'spectrum': color_item.get('spectrum'),
                'hue': color_item.get('hue'), 'percent': color_item.get('percent'), 'css3': color_item.get('css3')
            })

    return metadata_list, media_list, colors_list, None


# --- 3. Data Loading (Insert into SQL) ---
def insert_data_to_sql(metadata, media, colors):
    """Inserts the transformed data lists into the SQL database."""
    conn = sqlite3.connect(DB_NAME)

    def insert_batch(table_name, data_list):
        if not data_list: return 0
        try:
             pd.DataFrame(data_list).to_sql(table_name, conn, if_exists='append', index=False)
             return len(data_list)
        except Exception as e:
            return 0

    m_count = insert_batch('artifact_metadata', metadata)
    media_count = insert_batch('artifact_media', media)
    c_count = insert_batch('artifact_colors', colors)

    conn.close()
    return m_count, media_count, c_count

# --- 4. SQL Query Execution ---
def execute_sql_query(query):
    """Executes a SQL query and returns the results as a pandas DataFrame."""
    conn = sqlite3.connect(DB_NAME)
    try:
        df = pd.read_sql_query(query, conn)
        return df, None
    except pd.io.sql.DatabaseError as e:
        return None, str(e)
    finally:
        conn.close()

# --- 5. Predefined Queries List ---
def get_predefined_queries():
    # 20 required queries
    queries = {
        "1. Byzantine Artifacts (11th century)": "SELECT title, culture, century FROM artifact_metadata WHERE century = '11th century' AND culture = 'Byzantine'",
        "2. Unique Cultures": "SELECT DISTINCT culture FROM artifact_metadata WHERE culture IS NOT NULL",
        "3. Archaic Period Artifacts": "SELECT title, period FROM artifact_metadata WHERE period = 'Archaic Period'",
        "4. Titles by Accession Year (DESC)": "SELECT title, accessionyear FROM artifact_metadata ORDER BY accessionyear DESC LIMIT 10",
        "5. Artifacts per Department": "SELECT department, COUNT(id) AS ArtifactCount FROM artifact_metadata GROUP BY department",
        "6. Artifacts with > 1 Image": "SELECT T1.title FROM artifact_metadata AS T1 JOIN artifact_media AS T2 ON T1.id = T2.objectid WHERE T2.imagecount > 1",
        "7. Average Rank": "SELECT AVG(rank) AS AverageRank FROM artifact_media",
        "8. Colorcount > Mediacount": "SELECT T1.title, T2.colorcount, T2.mediacount FROM artifact_metadata AS T1 JOIN artifact_media AS T2 ON T1.id = T2.objectid WHERE T2.colorcount > T2.mediacount",
        "9. Artifacts created 1500-1600": "SELECT T1.title, T2.datebegin, T2.dateend FROM artifact_metadata AS T1 JOIN artifact_media AS T2 ON T1.id = T2.objectid WHERE T2.datebegin >= 1500 AND T2.dateend <= 1600",
        "10. Artifacts with No Media Files": "SELECT COUNT(T1.id) AS NoMediaCount FROM artifact_metadata AS T1 JOIN artifact_media AS T2 ON T1.id = T2.objectid WHERE T2.mediacount = 0",
        "11. Distinct Hues": "SELECT DISTINCT hue FROM artifact_colors WHERE hue IS NOT NULL",
        "12. Top 5 Most Used Colors": "SELECT color, COUNT(color) AS ColorFrequency FROM artifact_colors GROUP BY color ORDER BY ColorFrequency DESC LIMIT 5",
        "13. Average Coverage Percentage per Hue": "SELECT hue, AVG(percent) AS AverageCoveragePercent FROM artifact_colors GROUP BY hue",
        "14. List all colors used for a given artifact ID (Example: 2101).": "SELECT hue, color, percent FROM artifact_colors WHERE objectid = 2101",
        "15. What is the total number of color entries in the dataset?": "SELECT COUNT(color) AS TotalColorEntries FROM artifact_colors",
        "16. Titles and Hues for Byzantine Culture": """SELECT T1.title, T3.hue FROM artifact_metadata AS T1 JOIN artifact_colors AS T3 ON T1.id = T3.objectid WHERE T1.culture = 'Byzantine'""",
        "17. List each artifact title with its associated hues.": """SELECT T1.title, T2.hue FROM artifact_metadata AS T1 JOIN artifact_colors AS T2 ON T1.id = T2.objectid""",
        "18. Get artifact titles, cultures, and media ranks where the period is not null.": """SELECT T1.title, T1.culture, T2.rank FROM artifact_metadata AS T1 JOIN artifact_media AS T2 ON T1.id = T2.objectid WHERE T1.period IS NOT NULL""",
        "19. Find artifact titles ranked in the top 10 that include the color hue 'Grey'.": """SELECT DISTINCT T1.title, T2.rank FROM artifact_metadata AS T1 JOIN artifact_media AS T2 ON T1.id = T2.objectid JOIN artifact_colors AS T3 ON T1.id = T3.objectid WHERE T2.rank <= 10 AND T3.hue = 'Grey' ORDER BY T2.rank""",
        "20. Artifact Count & Avg Media Count per Classification": """SELECT T1.classification, COUNT(T1.id) AS ArtifactCount, AVG(T2.mediacount) AS AverageMediaCount FROM artifact_metadata AS T1 JOIN artifact_media AS T2 ON T1.id = T2.objectid GROUP BY T1.classification"""
    }
    return queries

# Ensure tables are created when this module is imported by Streamlit
create_tables()

Overwriting app_logic.py


In [10]:
%%writefile streamlit_app.py
import streamlit as st
import pandas as pd
from app_logic import (
    fetch_artifacts, insert_data_to_sql,
    execute_sql_query, get_predefined_queries
)

# --- Session State Initialization ---
if 'collected_metadata' not in st.session_state:
    st.session_state.collected_metadata = []
    st.session_state.collected_media = []
    st.session_state.collected_colors = []
    st.session_state.last_classification = ""
    st.session_state.api_error = None


# --- Streamlit UI Functions ---
def data_collection_section():
    """Handles the API fetching and SQL insertion UI."""
    st.header("1. 🎨 Data Collection (ETL)")
    st.markdown("Fetch, Clean, and Load a limited set of artifacts into a local SQLite database.")

    classification_options = [
        "Paintings", "Sculptures", "Coins", "Drawings", "Jewelry", "Ceramics", "Textiles"
    ]
    st.session_state.last_classification = st.selectbox(
        "Select an Artifact Classification",
        options=classification_options,
        index=0 # Default to Paintings
    )

    col1, col2, col3 = st.columns(3)

    with col1:
        # Button: Collect Data (E: Extract & T: Transform)
        if st.button("Collect Data (API)", key="collect_btn"):
            with st.spinner(f"Fetching data for '{st.session_state.last_classification}'..."):
                metadata, media, colors, error = fetch_artifacts(st.session_state.last_classification)

                if error:
                    st.error(f"Error during API fetch: {error}")
                    st.session_state.api_error = error
                elif metadata:
                    st.session_state.collected_metadata = metadata
                    st.session_state.collected_media = media
                    st.session_state.collected_colors = colors
                    st.session_state.api_error = None
                    st.success(f"✅ Extracted {len(metadata)} artifact records for '{st.session_state.last_classification}'.")
                else:
                    st.warning("No data returned from API. Check your classification name.")


    with col2:
        # Button: Show Data
        if st.button("Show Fetched Data Preview", key="show_btn"):
            if st.session_state.collected_metadata:
                st.subheader(f"Fetched Metadata ({len(st.session_state.collected_metadata)} records)")
                st.dataframe(pd.DataFrame(st.session_state.collected_metadata).head())
            else:
                st.warning("No data has been collected yet. Click 'Collect Data (API)'.")

    with col3:
        # Button: Insert into SQL (L: Load)
        if st.button("Insert into SQL (Load)", key="insert_btn"):
            if st.session_state.collected_metadata:
                with st.spinner("Inserting data into SQL tables..."):
                    m_count, media_count, c_count = insert_data_to_sql(
                        st.session_state.collected_metadata,
                        st.session_state.collected_media,
                        st.session_state.collected_colors
                    )
                    st.success(f"💾 Successfully loaded {m_count} artifacts, {media_count} media, and {c_count} color entries for '{st.session_state.last_classification}'.")
                    st.session_state.collected_metadata = []
            else:
                st.error("No data collected in memory to insert. Please collect data first.")


def sql_query_section():
    """Handles the SQL analytics and visualization UI."""
    st.header("2. 📊 SQL Analytics & Exploration")
    st.markdown("Run the 20 predefined SQL queries against the loaded database.")

    queries_dict = get_predefined_queries()
    query_description = st.selectbox(
        "Select a Pre-written Query (Total: 20)",
        options=list(queries_dict.keys())
    )

    selected_query = queries_dict[query_description]
    st.code(selected_query, language='sql')

    if st.button("Run Query", key="run_query_btn"):
        with st.spinner("Executing SQL Query..."):
            df, error = execute_sql_query(selected_query)

            if error:
                st.error(f"🚨 SQL Error: {error}")
                st.info("Ensure you have inserted data into the database before running queries.")
            elif df is not None and not df.empty:
                st.subheader("Query Results")
                st.dataframe(df)

                # Plot a chart for aggregated results
                if len(df.columns) >= 2 and any(col in df.columns[1] for col in ['Count', 'Average']):
                    st.bar_chart(df.set_index(df.columns[0]).iloc[:, 0])

            else:
                st.warning("Query executed successfully, but returned no results.")


# --- Main Streamlit App Layout ---
def main():
    st.set_page_config(layout="wide", page_title="Harvard Artifacts ETL & SQL Showcase")

    st.title("🏛️ Harvard's Artifacts Collection: ETL, SQL & Streamlit")
    st.markdown("An end-to-end data platform using the Harvard Art Museums API.")
    st.markdown("---")

    data_collection_section()
    st.markdown("---")
    sql_query_section()

if __name__ == '__main__':
    main()

Overwriting streamlit_app.py


In [11]:
# 3. Launch Streamlit with Ngrok (FINAL, CORRECTED VERSION)
from pyngrok import ngrok
import subprocess
import os
import time

# --- YOUR NGROK AUTH TOKEN ---
NGROK_AUTH_TOKEN = "343ENpNQteU9OTbMHQk8O5kdp67_4rJdcDMHDUX6ix8a3y18i"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
# -----------------------------

# Kill previous processes
print("Stopping previous processes...")
!pkill ngrok
!pkill streamlit

PORT = 8501
LOG_FILE = "streamlit.log"

print("Attempting to connect Ngrok tunnel...")

try:
    # 1. Start Ngrok tunnel first
    ngrok_tunnel = ngrok.connect(PORT)
    public_url = ngrok_tunnel.public_url

    # 2. Start the Streamlit app in the background with the CORRECT path './streamlit_app.py'
    print(f"Starting Streamlit app on port {PORT} with correct file path...")

    # Use python -m streamlit for robustness and explicitly pass './streamlit_app.py'
    subprocess.Popen(
        ['python', '-m', 'streamlit', 'run', './streamlit_app.py', # <--- FIXED PATH HERE
         '--server.port', str(PORT),
         '--server.enableCORS', 'false',
         '--server.enableXsrfProtection', 'false'],
        stdout=open(LOG_FILE, 'w'),
        stderr=open(LOG_FILE, 'a'),
        start_new_session=True
    )

    # 3. Wait and confirm
    time.sleep(15) # Increased wait time to ensure binding

    print(f"\n✅ Ngrok Tunnel Established and App Started!")
    print(f"Streamlit App Public URL: {public_url}")

    # Display log to confirm Streamlit started
    print("\nStreamlit logs (first few lines):")
    !cat {LOG_FILE} | head -n 10

    print("\n--- APP READY ---")
    print(f"1. Click the public URL: {public_url}")
    print("2. Proceed with ETL steps in the web app.")

except Exception as e:
    print(f"\n🚨 FAILED to start Ngrok or Streamlit.")
    print(f"Error details: {e}")

Stopping previous processes...
Attempting to connect Ngrok tunnel...
Starting Streamlit app on port 8501 with correct file path...

✅ Ngrok Tunnel Established and App Started!
Streamlit App Public URL: https://keli-unvillainous-progressionally.ngrok-free.dev

Streamlit logs (first few lines):
/usr/bin/python3: No module named streamlit

--- APP READY ---
1. Click the public URL: https://keli-unvillainous-progressionally.ngrok-free.dev
2. Proceed with ETL steps in the web app.


In [13]:
# 3. Launch Streamlit with Ngrok (FINAL, CORRECTED VERSION)
from pyngrok import ngrok
import subprocess
import os
import time

# --- YOUR NGROK AUTH TOKEN ---
NGROK_AUTH_TOKEN = "343ENpNQteU9OTbMHQk8O5kdp67_4rJdcDMHDUX6ix8a3y18i"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
# -----------------------------

# Kill previous processes
print("Stopping previous processes...")
!pkill ngrok
!pkill streamlit

PORT = 8501
LOG_FILE = "streamlit.log"

print("Attempting to connect Ngrok tunnel...")

try:
    # 1. Start Ngrok tunnel first
    ngrok_tunnel = ngrok.connect(PORT)
    public_url = ngrok_tunnel.public_url

    # 2. Start the Streamlit app in the background with the CORRECT path './streamlit_app.py'
    print(f"Starting Streamlit app on port {PORT} with correct file path...")

    # Use python -m streamlit for robustness and explicitly pass './streamlit_app.py'
    subprocess.Popen(
        ['python', '-m', 'streamlit', 'run', './streamlit_app.py', # <--- FIXED PATH HERE
         '--server.port', str(PORT),
         '--server.enableCORS', 'false',
         '--server.enableXsrfProtection', 'false'],
        stdout=open(LOG_FILE, 'w'),
        stderr=open(LOG_FILE, 'a'),
        start_new_session=True
    )

    # 3. Wait and confirm
    time.sleep(15) # Increased wait time to ensure binding

    print(f"\n✅ Ngrok Tunnel Established and App Started!")
    print(f"Streamlit App Public URL: {public_url}")

    # Display log to confirm Streamlit started
    print("\nStreamlit logs (first few lines):")
    !cat {LOG_FILE} | head -n 10

    print("\n--- APP READY ---")
    print(f"1. Click the public URL: {public_url}")
    print("2. Proceed with ETL steps in the web app.")

except Exception as e:
    print(f"\n🚨 FAILED to start Ngrok or Streamlit.")
    print(f"Error details: {e}")

Stopping previous processes...
Attempting to connect Ngrok tunnel...
Starting Streamlit app on port 8501 with correct file path...

✅ Ngrok Tunnel Established and App Started!
Streamlit App Public URL: https://keli-unvillainous-progressionally.ngrok-free.dev

Streamlit logs (first few lines):



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.169.42.93:8501


--- APP READY ---
1. Click the public URL: https://keli-unvillainous-progressionally.ngrok-free.dev
2. Proceed with ETL steps in the web app.


In [ ]:
# 3. Launch Streamlit with Ngrok (FINAL, CORRECTED VERSION)
from pyngrok import ngrok
import subprocess
import os
import time

# --- YOUR NGROK AUTH TOKEN ---
NGROK_AUTH_TOKEN = "343ENpNQteU9OTbMHQk8O5kdp67_4rJdcDMHDUX6ix8a3y18i"
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
# -----------------------------

# Kill previous processes
print("Stopping previous processes...")
!pkill ngrok
!pkill streamlit

PORT = 8501
LOG_FILE = "streamlit.log"

print("Attempting to connect Ngrok tunnel...")

try:
    # 1. Start Ngrok tunnel first
    ngrok_tunnel = ngrok.connect(PORT)
    public_url = ngrok_tunnel.public_url

    # 2. Start the Streamlit app in the background with the CORRECT path './streamlit_app.py'
    print(f"Starting Streamlit app on port {PORT} with correct file path...")

    # Use python -m streamlit for robustness and explicitly pass './streamlit_app.py'
    subprocess.Popen(
        ['python', '-m', 'streamlit', 'run', './streamlit_app.py', # <--- FIXED PATH HERE
         '--server.port', str(PORT),
         '--server.enableCORS', 'false',
         '--server.enableXsrfProtection', 'false'],
        stdout=open(LOG_FILE, 'w'),
        stderr=open(LOG_FILE, 'a'),
        start_new_session=True
    )

    # 3. Wait and confirm
    time.sleep(15) # Increased wait time to ensure binding

    print(f"\n✅ Ngrok Tunnel Established and App Started!")
    print(f"Streamlit App Public URL: {public_url}")

    # Display log to confirm Streamlit started
    print("\nStreamlit logs (first few lines):")
    !cat {LOG_FILE} | head -n 10

    print("\n--- APP READY ---")
    print(f"1. Click the public URL: {public_url}")
    print("2. Proceed with ETL steps in the web app.")

except Exception as e:
    print(f"\n🚨 FAILED to start Ngrok or Streamlit.")
    print(f"Error details: {e}")

In [12]:
!pip install streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 118.1 MB/s eta 0:00:00
